In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import (
    KMeans,
    DBSCAN,
    Birch,
    AgglomerativeClustering,
    SpectralClustering,
    MeanShift
)
from sklearn.cluster import estimate_bandwidth
from sklearn.mixture import (
    GaussianMixture,
    BayesianGaussianMixture
)
from sklearn.decomposition import PCA

from multiprocessing import Process, Queue, SimpleQueue
import time

In [2]:
full = np.load("../data/processed/FULL_30.npz", allow_pickle=True)
X_full  = full["X"]#[::10]#[::10]
K0_full = full["K0"]#[::10]#[::10]

In [3]:
def run_model(queue, model, X):

    try:
        print(
            f"START fit_predict "
            f"{model.__class__.__name__} "
            f"N={len(X)}",
            flush=True
        )

        t0 = time.perf_counter()

        labels = model.fit_predict(X)

        print(
            f"END fit_predict "
            f"{time.perf_counter() - t0:.2f}s",
            flush=True
        )

        print(
            "START queue.put",
            flush=True
        )
        print(
            labels.dtype,
            labels.shape,
            labels.nbytes / 1024**2,
            "MB",
            flush=True
        )
        queue.put(labels)
        # queue.put(1)

        print(
            "END queue.put",
            flush=True
        )

    except Exception as e:
        print(
            f"EXCEPTION: {e}",
            flush=True
        )
        queue.put(e)

In [4]:
import queue as queue_mod

def fit_predict_timeout(model, X, timeout_sec):

    q = Queue()

    p = Process(
        target=run_model,
        args=(q, model, X)
    )

    p.start()

    try:
        # najpierw odbierz wynik
        result = q.get(timeout=timeout_sec)

    except queue_mod.Empty:

        p.terminate()
        p.join()

        return None, "timeout"

    # dopiero potem czekaj na proces
    p.join()

    if isinstance(result, Exception):
        return None, str(result)

    return result, None

In [5]:
TIME_LIMITS = {
    250: 60,    # 1 min
    100: 300,   # 5 min
    # 50: 600,    # 10 min
    # 10: 1200,    # 20 min
}
strides = [250,100]
# strides = [50,10]

In [6]:
from itertools import combinations
import numpy as np

cols_20 = np.setdiff1d(
    np.arange(30),
    [2,3,5,7,12,18,21,24,27,28]
)

removed = [2,3,5,7,12,18,21,24,27,28]
candidates = [2,5,12,24]

feature_sets = {
    "full_20": cols_20
}

for r in range(1, len(candidates) + 1):
    for combo in combinations(candidates, r):

        removed_new = [x for x in removed if x not in combo]

        feature_sets[f"add_{'_'.join(map(str, combo))}"] = (
            np.setdiff1d(np.arange(30), removed_new)
        )

for name, cols in feature_sets.items():
    print(name, len(cols))

full_20 20
add_2 21
add_5 21
add_12 21
add_24 21
add_2_5 22
add_2_12 22
add_2_24 22
add_5_12 22
add_5_24 22
add_12_24 22
add_2_5_12 23
add_2_5_24 23
add_2_12_24 23
add_5_12_24 23
add_2_5_12_24 24


In [7]:
feature_sets

{'full_20': array([ 0,  1,  4,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20, 22, 23,
        25, 26, 29]),
 'add_2': array([ 0,  1,  2,  4,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20, 22,
        23, 25, 26, 29]),
 'add_5': array([ 0,  1,  4,  5,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20, 22,
        23, 25, 26, 29]),
 'add_12': array([ 0,  1,  4,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22,
        23, 25, 26, 29]),
 'add_24': array([ 0,  1,  4,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20, 22, 23,
        24, 25, 26, 29]),
 'add_2_5': array([ 0,  1,  2,  4,  5,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20,
        22, 23, 25, 26, 29]),
 'add_2_12': array([ 0,  1,  2,  4,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20,
        22, 23, 25, 26, 29]),
 'add_2_24': array([ 0,  1,  2,  4,  6,  8,  9, 10, 11, 13, 14, 15, 16, 17, 19, 20, 22,
        23, 24, 25, 26, 29]),
 'add_5_12': array([ 0,  1,  4,  5,  6,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20,
    

In [8]:
# feature_sets = {
#     "30": np.arange(30),
#     "21": cols_21,
#     "12": cols_12,
# }

In [9]:
summary_rows = []
mean_rows = []
chi_rows = []

In [10]:
def stat_mean(x):
    return np.mean(x)



def stat_susceptibility(x):
    mu = np.mean(x)

    if abs(mu) < 1e-12:
        return 0.0

    return np.var(x, ddof=1) #######/ mu

    
    # def stat_susceptibility(x):
#     # Mathematica: Variance[...] / Mean[...] where Variance uses ddof=1
#     return np.var(x, ddof=1) / np.mean(x)

In [11]:
def block_jackknife(data, stat_func, B):
    """
    Block jackknife with B blocks.
      - truncate data to m*B elements
      - reshape into B blocks of size m
      - compute full estimator EX on truncated data
      - leave one block out at a time, compute stat on remaining B-1 blocks
      - bias = (B-1) * (mean(dats) - EX)          [jack.pdf eq 3.2.1]
      - theta_jack = EX - bias                      [bias-corrected estimate]
      - var = Variance(dats) * (B-1)^2 / B         [Mathematica Variance = ddof=1]
      - se = sqrt(var)

    Returns: (theta_hat, theta_jack, se) for this specific B,
             or None if block size < 1.
    """
    n = len(data)
    m = n // B  # block size (integer division, matches Mathematica Quotient)

    if m < 1:
        return None

    # Truncate to m*B  (matches: data[[1 ;; Quotient[len,B]*B]])
    data = data[:m * B]

    # Reshape into B blocks  (matches: Table[data[[m*i+1 ;; m*(i+1)]], {i,0,B-1}])
    blocks = data.reshape(B, m)

    # Full estimator on truncated data  (matches: EX = stat_func(data))
    theta_hat = stat_func(data)

    # Leave-one-block-out replicates
    # Matches: rands = Table[DeleteCases[Range[B], i], {i, B}]
    #          dats  = Table[stat(Flatten[blocks[rands[i]]]), {i, B}]
    dats = np.array([
        stat_func(np.concatenate([blocks[:i], blocks[i+1:]]).ravel())
        for i in range(B)
    ])

    # Bias: (B-1) * (mean(dats) - EX)  [jack.pdf 3.2.1, Mathematica bias line]
    bias = (B - 1) * (np.mean(dats) - theta_hat)

    # Bias-corrected estimator: EXUn = EX - bias
    theta_jack = theta_hat - bias

    # Variance of replications using ddof=1  (Mathematica Variance = sample variance)
    # var = Variance[dats] * (B-1)^2 / B
    var = np.var(dats, ddof=1) * (B - 1)**2 / B
    se = np.sqrt(var)

    return theta_hat, theta_jack, se



In [12]:
# =============================================================================
# JACKKNIFE OVER K0 — returns FULL TABLE for every B, matching Mathematica output
# =============================================================================

def jackknife_over_K(P_A, K0, stat_func, B_range=range(2, 101)):
    """
    For each unique K0 value, run block_jackknife for every B in B_range.

    Returns a dict:  {k: DataFrame with columns [B, theta_hat, theta_jack, se]}

    This matches the Mathematica Table[..., {ildanych, 2, 100, 1}] output exactly.
    You can then inspect SE vs B plots to find the plateau, rather than
    artificially picking the maximum SE.
    """
    df_input = pd.DataFrame({"K0": K0, "P_A": P_A})
    results = {}

    for k, group in df_input.groupby("K0"):
        data = group["P_A"].values
        # print(k, len(data))

        if len(data) < 2:
            continue

        rows = []
        for B in B_range:
            out = block_jackknife(data, stat_func, B)
            if out is None:
                continue
            theta_hat, theta_jack, se = out
            rows.append({
                "B":          B,
                "theta_hat":  theta_hat,   # EX  in Mathematica
                "theta_jack": theta_jack,  # EXUn in Mathematica
                "se":         se,          # Smse in Mathematica
            })

        results[k] = pd.DataFrame(rows)
    return results

In [13]:
def summarise_jackknife(results_dict):

    K_vals, theta_arr, se_arr = [], [], []

    for k in sorted(results_dict.keys()):
        df = results_dict[k]

        if df.empty:
            continue

        df_valid = df.dropna(subset=["se"])

        if df_valid.empty:
            print(f"All SE are NaN for K0={k}")
            continue

        row = df_valid.loc[df_valid["se"].idxmax()]

        K_vals.append(k)
        theta_arr.append(row["theta_jack"])
        se_arr.append(row["se"])

    return (
        np.array(K_vals),
        np.array(theta_arr),
        np.array(se_arr),
    )



def plot_se_vs_B(results_dict, stat_label, target_Ks=None, ncols=5):
    """
    Plot SE vs B for each K value.
    Use this to verify that the SE has plateaued before trusting the summary.
    All K values are shown, arranged in a grid with ncols columns.
    """
    ks = sorted(results_dict.keys())
    if target_Ks is not None:
        ks = [k for k in ks if k in target_Ks]

    n = len(ks)
    if n == 0:
        return

    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

    for idx, k in enumerate(ks):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]
        df = results_dict[k]
        ax.plot(df["B"], df["se"], lw=1.5)
        ax.set_title(f"K0 = {k:.3f}")
        ax.set_xlabel("B (number of blocks)")
        ax.set_ylabel(f"SE of {stat_label}")
        ax.grid(True, alpha=0.3)

    # Hide any leftover empty subplots
    for idx in range(n, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    fig.suptitle(f"SE vs B — {stat_label} (should plateau)", y=1.02)
    plt.tight_layout()

In [14]:
from scipy.interpolate import interp1d
from scipy.optimize import brentq


def find_k0_crit(K, mean, mean_err):
    """
    Find K where mean(K)=0.5
    and estimate uncertainty via error propagation.
    """

    idx_close = np.argmin(np.abs(mean-0.5))
    k_close = K[idx_close]
    
    f = interp1d(
        K,
        mean - 0.5,
        kind="linear",
        bounds_error=False
    )

    idx = np.where(
        (mean[:-1] - 0.5) *
        (mean[1:]  - 0.5) <= 0
    )[0]

    if len(idx) == 0:
        return np.nan, np.nan, np.nan

    i = idx[0]

    kcrit = brentq(
        lambda x: f(x),
        K[i],
        K[i + 1]
    )

    slope = (
        mean[i + 1] - mean[i]
    ) / (
        K[i + 1] - K[i]
    )

    sigma_p = np.interp(
        kcrit,
        [K[i], K[i + 1]],
        [mean_err[i], mean_err[i + 1]]
    )
    if abs(slope) < 1e-12:
        return kcrit, np.nan, k_close
    
    sigma_k = sigma_p / abs(slope)

    return kcrit, sigma_k, k_close


In [15]:
def timed(func, *args, **kwargs):
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    return result, elapsed

In [16]:
from multiprocessing import Process, Queue

def run_gmm(queue, model, X):

    try:
        model.fit(X)

        probs = model.predict_proba(X)

        queue.put(
            (
                probs,
                model.means_,
                model.weights_,
                model.converged_,
                model.n_iter_,
            )
        )

    except Exception as e:
        queue.put(e)


def fit_gmm_timeout(model, X, timeout_sec):

    q = Queue()

    p = Process(
        target=run_gmm,
        args=(q, model, X)
    )

    p.start()

    p.join(timeout_sec)

    if p.is_alive():
        p.terminate()
        p.join()
        return None, "timeout"

    if q.empty():
        return None, "empty queue"

    result = q.get()

    if isinstance(result, Exception):
        return None, str(result)

    return result, None

In [17]:

# =============================================================================
# MAIN
# =============================================================================

models = {
    "KMeans": lambda: KMeans(
        n_clusters=2,
        random_state=0
    ),

    "Agglomerative": lambda: AgglomerativeClustering(
    n_clusters=2,
    # connectivity=connectivity
    ),

    "Spectral": lambda: SpectralClustering(
        n_clusters=2,
        affinity="nearest_neighbors",
        random_state=0
    ),

    # "GaussianMixture": lambda: GaussianMixture(
    #     n_components=2,
    #     random_state=0,
    #     max_iter=1000
    # ),

    # "MeanShift": lambda: MeanShift(
    #     # bandwidth=bandwidth_meanshift
    # ),

    # "DBSCAN": lambda: DBSCAN(
    #     # eps=50,
    #     # min_samples=10
    # ),

    "Birch": lambda: Birch(
        n_clusters=2
    ),

    # poza artykułem
    # "BayesianGMM": lambda: BayesianGaussianMixture(
    #     n_components=2,
    #     random_state=0,
    #     max_iter=3000

    # ),
}

curves = {}

for feature_name, cols in feature_sets.items():

    X_full_fs  = X_full[:, cols]

    curves[feature_name] = {}

    print(
        f"\n{'='*70}"
        f"\nFEATURE SET: {feature_name}"
        f" ({len(cols)} features)"
        f"\n{'='*70}"
    )

    t0 = time.perf_counter()
    for name, model_factory in models.items():
        print(f"\n{'='*50}\nModel: {name}\n{'='*50}")
        curves[feature_name][name] = {}
    

        for stride in strides:
        
            timeout_sec = TIME_LIMITS[stride]
        
            X_try = X_full_fs[::stride]
            K0_try = K0_full[::stride]
        
            print(
                f"Trying stride={stride}, "
                f"N={len(X_try)}, "
                f"timeout={timeout_sec}s"
            )
        
            model_instance = model_factory()

            t0 = time.perf_counter()
            if name in ["GaussianMixture", "BayesianGMM"]:
            
                gmm_result, err = fit_gmm_timeout(
                    model_instance,
                    X_try,
                    timeout_sec
                )
            
                if err is not None:
                    print(f"stride={stride} failed: {err}")

                    if err == "timeout":
                        print("Stopping smaller strides")
                        break
                
                    continue
                (
                    probs,
                    means,
                    weights,
                    converged,
                    n_iter
                ) = gmm_result
            
                print(
                    f"converged={converged}, "
                    f"iterations={n_iter}"
                )
            
                phase_A = np.argmax(means[:, 0])
            
                P_A = probs[:, phase_A]
            
                K0_used = K0_try
            
            else:
            
                labels, err = fit_predict_timeout(
                    model_instance,
                    X_try,
                    timeout_sec
                )
            
                if err is not None:
                    print(f"stride={stride} failed: {err}")

                    if err == "timeout":
                        print("Stopping smaller strides")
                        break
                
                    continue
            
                valid = labels != -1
            
                labels = labels[valid]
                K0_used = K0_try[valid]
            
                if len(np.unique(labels)) < 2:
                    print("Only one cluster")
                    continue
            
                df = pd.DataFrame({
                    "label": labels,
                    "K0": K0_used
                })
            
                cluster_means = (
                    df.groupby("label")["K0"]
                      .mean()
                )
            
                phase_A = cluster_means.idxmax()
            
                P_A = (
                    labels == phase_A
                ).astype(float)
                    
            model_time = time.perf_counter() - t0
        
            # -----------------------
            # jackknife
            # -----------------------

            print("starting jackknife")
            mean_results, mean_jack_time = timed(
                jackknife_over_K,
                P_A,
                K0_used,
                stat_mean,
            )
            print("finishing jackknife")
        
            chi_results, chi_jack_time = timed(
                jackknife_over_K,
                P_A,
                K0_used,
                stat_susceptibility,
            )
        
            if len(mean_results) == 0:
                continue
        
            (K_mean, mean_jack, mean_se), summary_mean_time = timed(
                summarise_jackknife,
                mean_results,
            )
        
            (K_chi, chi_jack, chi_se), summary_chi_time = timed(
                summarise_jackknife,
                chi_results,
            )
        
            if len(K_mean) == 0:
                continue
        
            (kcrit, kcrit_err, k_close), time_k0_crit = timed(
                find_k0_crit,
                K_mean,
                mean_jack,
                mean_se,
            )
        
            total_time = (model_time + 
                mean_jack_time
                + chi_jack_time
                + summary_mean_time
                + summary_chi_time
                + time_k0_crit
            )
        
            print(
                f"{name:20s} | stride={stride} | total={total_time:.2f}s"
            )
        
            summary_rows.append({
                "stride": stride,
                "feature_set": feature_name,
                "n_features": len(cols),
                "model": name,
                "K0_crit": kcrit,
                "K0_crit_err": kcrit_err,
                "K0_close": k_close,
                "N_K0": len(K_mean),
                "runtime_model": round(model_time, 3),
                "runtime_jackknife_mean": round(mean_jack_time, 3),
                "runtime_jackknife_chi": round(chi_jack_time, 3),
                "summary_mean_time": round(summary_mean_time, 4),
                "summary_chi_time": round(summary_chi_time, 4),
                "runtime_k0_crit": round(time_k0_crit, 5),
                "runtime_total": round(total_time, 3),        
            })
        
            for k, m, err in zip(
                K_mean,
                mean_jack,
                mean_se
            ):
                mean_rows.append({
                    "stride": stride,
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": name,
                    "K0": k,
                    "mean": m,
                    "mean_err": err,
                })
        
            for k, c, err in zip(
                K_chi,
                chi_jack,
                chi_se
            ):
                chi_rows.append({
                    "stride": stride,
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": name,
                    "K0": k,
                    "chi": c,
                    "chi_err": err,
                })
    



FEATURE SET: full_20 (20 features)

Model: KMeans
Trying stride=250, N=6006, timeout=60s
START fit_predict KMeans N=6006
END fit_predict 0.16s
START queue.put
int32 (6006,) 0.02291107177734375 MB
END queue.put
starting jackknife
finishing jackknife
KMeans               | stride=250 | total=5.06s
Trying stride=100, N=15015, timeout=300s
START fit_predict KMeans N=15015
END fit_predict 0.19s
START queue.put
int32 (15015,) 0.057277679443359375 MB
END queue.put
starting jackknife
finishing jackknife
KMeans               | stride=100 | total=5.42s

Model: Agglomerative
Trying stride=250, N=6006, timeout=60s
START fit_predict AgglomerativeClustering N=6006
END fit_predict 1.12s
START queue.put
int64 (6006,) 0.0458221435546875 MB
END queue.put
starting jackknife
finishing jackknife
Agglomerative        | stride=250 | total=6.09s
Trying stride=100, N=15015, timeout=300s
START fit_predict AgglomerativeClustering N=15015
END fit_predict 7.74s
START queue.put
int64 (15015,) 0.11455535888671875 M

In [19]:
import os

# os.makedirs("results", exist_ok=True)

pd.DataFrame(summary_rows).to_csv(
    "./20feat_comparison/summary.csv",
    index=False
)

pd.DataFrame(mean_rows).to_csv(
    "./20feat_comparison/curve_mean.csv",
    index=False
)

pd.DataFrame(chi_rows).to_csv(
    "./20feat_comparison/curve_chi.csv",
    index=False
)

In [ ]:
# labels = DBSCAN(
#     eps=0.9,
#     min_samples=10
# ).fit_predict(X_full)

# print(np.unique(labels, return_counts=True))

In [ ]:
# for eps in [0.2, 5, 20,1,5,10,100,500,1000,5000,7500,10000, 20000,50000]:
#     labels = DBSCAN(
#         eps=eps,
#         min_samples=10
#     ).fit_predict(X_pca_scaled)

#     print(
#         eps,
#         np.unique(labels, return_counts=True)
#     )

metody wymuszające podział (KMeans, GMM, Agglomerative) znajdują granicę,
metody gęstościowe (DBSCAN, MeanShift) mówią: "to jest jeden obiekt".

In [ ]:
# from sklearn.cluster import KMeans
# import time

# X_try = X_full[::10]

# t0 = time.perf_counter()

# km = KMeans(
#     n_clusters=2,
#     random_state=0
# )

# labels = km.fit_predict(X_try)

# print(
#     "runtime:",
#     time.perf_counter() - t0
# )

In [ ]:
# X_try = X_full[:, :]
# labels = KMeans(
#     n_clusters=2,
#     random_state=0
# ).fit_predict(X_try)

# print(labels.dtype)
# print(labels.shape)
# print(labels.nbytes)